In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os 
from datetime import datetime

import hyperparameters
from utils import RandomTrial, read_data

In [2]:
from modelsnnpc_withproba import *

In [3]:
hyperparameters.P20_S50

{'population': 20,
 'batch': 1024,
 'epoch': 5,
 'step': 50,
 'beta': (0.7504599493, 0.8773360581, 0.2536912312, 0.9376188938),
 'slope': 12,
 'threshold': (0.3261930083, 0.8492504097, 0.4361259464, 0.8395399557),
 'weight': 0.9743752082,
 'adam_beta': (0.978, 0.974),
 'learning_rate': 0.0007997609}

In [4]:
DATASET_LIST = ["Variant I"]
# DATASET_LIST = ["Base", "Variant I", "Variant II", "Variant III", "Variant IV", "Variant V"]
NUM_TRIALS = 10
BEGIN_TRIAL = 0
BASE_SEED = 42

METRICS_NAME_GLOBAL = ["accuracy", "precision", "recall", "fpr", "f1_score","auc"]
METRICS_NAME_5FPR = ["accuracy@5FPR","precision@5FPR", "recall@5FPR", "fpr@5FPR", "f1_score@5FPR"]
METRICS_FAIRNESS = ["fpr_ratio_age", "fpr_ratio_income", "fpr_ratio_employment",
                   "eod_age", "eod_income", "eod_employment",
                   "aod_age", "aod_income", "aod_employment"]
HYPERPARAMETERS = hyperparameters.P20_S50 

EXPERIMENT_NAME = f"P{HYPERPARAMETERS['population']}-S{HYPERPARAMETERS['step']}-{NUM_TRIALS}trials-begin{BEGIN_TRIAL}"
FIXED_DATE = None

In [5]:
def dataset_loop(train_dfs, test_dfs, dataset_name, trial_number, seed, path, runs):
    x_train = train_dfs[dataset_name].drop(columns=["fraud_bool"])
    x_train["customer_age"] = 0
    y_train = train_dfs[dataset_name]["fraud_bool"]
    x_test = test_dfs[dataset_name].drop(columns=["fraud_bool"])
    x_test["customer_age"] = 0
    y_test = test_dfs[dataset_name]["fraud_bool"]
    num_classes = len(np.unique(y_train))
    num_features = len(x_train.columns)
    class_weights = (1-HYPERPARAMETERS['weight'], HYPERPARAMETERS['weight'])
    model = ModelSNNPC(
        num_features=num_features,
        num_classes=num_classes,
        class_weights=class_weights,
        betas=HYPERPARAMETERS['beta'],
        slope=HYPERPARAMETERS['slope'],
        thresholds=HYPERPARAMETERS['threshold'],
        population=HYPERPARAMETERS['population'],
        batch_size=HYPERPARAMETERS['batch'],
        num_epochs=HYPERPARAMETERS['epoch'],
        num_steps=HYPERPARAMETERS['step'],
        adam_betas=HYPERPARAMETERS['adam_beta'],
        learning_rate=HYPERPARAMETERS['learning_rate'],
        verbose=1
    )
    model.fit(x_train, y_train)
    predictions, targets, proba = model.predict(x_test, y_test)
    np.savetxt(f"predictions_run{trial_number}.csv", predictions, delimiter=",", fmt="%d") 
    np.savetxt(f"targets_run{trial_number}.csv", targets, delimiter=",", fmt="%d")
    np.savetxt(f"proba_run{trial_number}.csv", proba, delimiter="\t")
    metrics = model.evaluate(targets, predictions)
    metrics_aequitas = model.evaluate_business_constraint(targets, predictions)
    metrics.update(metrics_aequitas)
    fairness_age = model.evaluate_fairness(x_test, targets, predictions, "customer_age", -1)
    metrics.update({k+"_age": v for k, v in fairness_age.items()})
    fairness_income = model.evaluate_fairness(x_test, targets, predictions, "income", 0.5)
    metrics.update({k+"_income": v for k, v in fairness_income.items()})
    fairness_employement = model.evaluate_fairness(x_test, targets, predictions, "employment_status", 3)
    metrics.update({k+"_employment": v for k, v in fairness_employement.items()})
    results = {}
    results["dataset"] = dataset_name
    results["trial"] = trial_number
    results["seed"] = seed
    for metric in METRICS_NAME_GLOBAL:
        results[metric] = metrics[metric]
    for metric in METRICS_NAME_5FPR:
        results[metric] = metrics_aequitas[metric]
    for metric in METRICS_FAIRNESS:
        results[metric] = metrics[metric]
    csv_row = ','.join([str(x) for x in results.values()])
    with open(path, "a") as f:
        f.write(f"{csv_row}\n")
    prev_runs = runs.get(dataset_name, [])
    prev_runs.append(results)
    print(results)
    runs[dataset_name] = prev_runs
    return runs


In [6]:
base_path = "/kaggle/input/bank-account-fraud-dataset/"
_, datasets, train_dfs, test_dfs = read_data(base_path, DATASET_LIST)
if not FIXED_DATE:
    date = datetime.now().strftime("%Y%m%d_%H%M%S")
else:
    date = FIXED_DATE
experiment_dir = f"/kaggle/working/results/{date}-{EXPERIMENT_NAME}"
results_path = f"{experiment_dir}/results.csv"
os.makedirs(experiment_dir, exist_ok=True)
if not os.path.exists(results_path):
    with open(results_path, "w") as f:
        f.write("dataset,trial,seed,accuracy,precision,recall,fpr,f1_score,auc,accuracy@5FPR,precision@5FPR,recall@5FPR,fpr@5FPR,f1_score@5FPR,fpr_ratio_age,fpr_ratio_income,fpr_ratio_employment,eod_age,eod_income,eod_employment,aod_age,aod_income,aod_employment\n")

In [7]:
def simulation(datasets, train_dfs, test_dfs, path="./results.csv"):
    np.random.seed(BASE_SEED)
    seeds = np.random.choice(list(range(1_000_000)), size=NUM_TRIALS, replace=False)
    runs = {}
    for trial in range(NUM_TRIALS):
        seed = seeds[trial]
        trial_number = trial
        trial = RandomTrial(seed=seed)
        if trial_number < BEGIN_TRIAL:
            print(f"Skipping trial {trial_number} – seed {seed}")
            continue
        for dataset_name in datasets.keys():
            print(f"Running trial {trial_number} with seed {seed} on dataset {dataset_name}")
            runs = dataset_loop(train_dfs, test_dfs, dataset_name, trial_number, seed, path, runs)
    return runs

In [8]:
simulation(datasets, train_dfs, test_dfs, path=results_path)

Running trial 0 with seed 987231 on dataset Variant I
{'dataset': 'Variant I', 'trial': 0, 'seed': 987231, 'accuracy': 0.9169894151504805, 'precision': 0.08876221498371335, 'recall': 0.5302293259207783, 'fpr': 0.07750380939188253, 'f1_score': 0.15206776283009465, 'auc': 0.7263627582644478, 'accuracy@5FPR': 0.9859616604068094, 'precision@5FPR': 0, 'recall@5FPR': 0.0, 'fpr@5FPR': 0.0, 'f1_score@5FPR': 0, 'fpr_ratio_age': None, 'fpr_ratio_income': 0, 'fpr_ratio_employment': 0, 'eod_age': None, 'eod_income': 0.0, 'eod_employment': 0.0, 'aod_age': None, 'aod_income': 0.0, 'aod_employment': 0.0}
Running trial 1 with seed 79954 on dataset Variant I
{'dataset': 'Variant I', 'trial': 1, 'seed': 79954, 'accuracy': 0.9573581776498707, 'precision': 0.13313313313313313, 'recall': 0.36970118137595553, 'fpr': 0.034274632418419645, 'f1_score': 0.19576816927322907, 'auc': 0.6677132744787679, 'accuracy@5FPR': 0.9573581776498707, 'precision@5FPR': 0.13313313313313313, 'recall@5FPR': 0.36970118137595553, 

{'Variant I': [{'dataset': 'Variant I',
   'trial': 0,
   'seed': 987231,
   'accuracy': 0.9169894151504805,
   'precision': 0.08876221498371335,
   'recall': 0.5302293259207783,
   'fpr': 0.07750380939188253,
   'f1_score': 0.15206776283009465,
   'auc': 0.7263627582644478,
   'accuracy@5FPR': 0.9859616604068094,
   'precision@5FPR': 0,
   'recall@5FPR': 0.0,
   'fpr@5FPR': 0.0,
   'f1_score@5FPR': 0,
   'fpr_ratio_age': None,
   'fpr_ratio_income': 0,
   'fpr_ratio_employment': 0,
   'eod_age': None,
   'eod_income': 0.0,
   'eod_employment': 0.0,
   'aod_age': None,
   'aod_income': 0.0,
   'aod_employment': 0.0},
  {'dataset': 'Variant I',
   'trial': 1,
   'seed': 79954,
   'accuracy': 0.9573581776498707,
   'precision': 0.13313313313313313,
   'recall': 0.36970118137595553,
   'fpr': 0.034274632418419645,
   'f1_score': 0.19576816927322907,
   'auc': 0.6677132744787679,
   'accuracy@5FPR': 0.9573581776498707,
   'precision@5FPR': 0.13313313313313313,
   'recall@5FPR': 0.369701181